In [ ]:
!pip install pymilvus tqdm requests

In [2]:
import json
from pathlib import Path

CHUNKS_PATH = Path("chunks.json")  # adjust path if it's in a different folder

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} chunks")

Loaded 1045 chunks


In [7]:
from dotenv import load_dotenv
import os

load_dotenv()

ZILLIZ_URI = os.getenv("ZILLIZ_URI")
ZILLIZ_TOKEN = os.getenv("ZILLIZ_TOKEN")
DR_COLLECTION_NAME = os.getenv("DR_COLLECTION_NAME", "altibbi_doctors")
EMBEDDING_URL = os.getenv("EMBEDDING_URL")
EMBEDDING_API_KEY = os.getenv("EMBEDDING_API_KEY")
EMBED_DIM         = 768
BATCH_SIZE        = 16

In [8]:
from pymilvus import connections, Collection, CollectionSchema, FieldSchema, DataType


# connect to ZILLIZ DB
connections.connect(uri=ZILLIZ_URI, token=ZILLIZ_TOKEN)
print("Connected to Zilliz")

# defines a table structure — matches the doctor chunk metadata
# (doctor_id is a string in the source data, e.g. "96478", so VARCHAR not INT64)
fields = [
    FieldSchema(name="id",          dtype=DataType.INT64,       is_primary=True, auto_id=True),
    FieldSchema(name="doctor_id",   dtype=DataType.VARCHAR,     max_length=50),
    FieldSchema(name="chunk_index", dtype=DataType.INT64),
    FieldSchema(name="text",        dtype=DataType.VARCHAR,     max_length=2000),
    FieldSchema(name="name",        dtype=DataType.VARCHAR,     max_length=200),
    FieldSchema(name="specialty",   dtype=DataType.VARCHAR,     max_length=300),
    FieldSchema(name="location",    dtype=DataType.VARCHAR,     max_length=500),
    FieldSchema(name="url",         dtype=DataType.VARCHAR,     max_length=500),
    FieldSchema(name="embedding",   dtype=DataType.FLOAT_VECTOR, dim=EMBED_DIM),
]

# Create a collection named altibbi_doctors with this structure
schema = CollectionSchema(fields, description="Altibbi doctor profile chunks (Arabic)")
col = Collection(name=DR_COLLECTION_NAME, schema=schema)

# Create an index: It builds a structure to make search fast.
col.create_index(
    field_name="embedding",
    index_params={"metric_type": "COSINE", "index_type": "AUTOINDEX"}
)

# Load the collection
col.load()  # Milvus prepares it in a search-ready memory layer -> fast to run similarity search
print("Collection created")

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_23044\1486779966.py:5: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  connections.connect(uri=ZILLIZ_URI, token=ZILLIZ_TOKEN)


Connected to Zilliz


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_23044\1486779966.py:24: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  col = Collection(name=DR_COLLECTION_NAME, schema=schema)
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_23044\1486779966.py:27: PyMilvusDeprecationWarning: `Collection.create_index` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  col.create_index(
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_23044\1486779966.py:33: PyMilvusDeprecationWarning: `Collection.load` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  col.load()  # Milvus prepares it in a search-ready memory layer -> fast to run similarity search


Collection created


In [11]:
import requests
import time
from tqdm import tqdm

def get_embeddings(texts, retries=3):
    """
    sends text to embedding API and returns embedding vectors
    """
    headers = {"Authorization": EMBEDDING_API_KEY, "Content-Type": "application/json"}
    # what is sent
    payload = {
        "docs": texts,           # list of text chunks
        "dense_weight": 1.0,     # semantic search
        "sparse_weight": 0.0,    # traditional search (tf-idf)
        "convert_to_float32": True,  # reduces memory usage
        "normalize_vectors": False,
        "batch_size": BATCH_SIZE,
    }

    # retry 3 times if api fails
    for attempt in range(retries):
        try:
            resp = requests.post(EMBEDDING_URL, json=payload, headers=headers, timeout=60)  # api call
            resp.raise_for_status()
            data = resp.json()
            return data["embeddings"]
        except Exception as e:
            wait = 2 ** attempt
            print(f"Attempt {attempt+1} failed: {e}. Retrying in {wait}s...")
            time.sleep(wait)
    raise RuntimeError("Embedding API failed.")

total = len(chunks)
for batch_start in tqdm(range(0, total, BATCH_SIZE), desc="Embedding"):
    batch = chunks[batch_start: batch_start + BATCH_SIZE]
    texts = [c["text"] for c in batch]  # extract only text (not metadata)

    try:
        embeddings = get_embeddings(texts)
    except RuntimeError as e:
        print(f"Failed at batch {batch_start}: {e}")
        break

    rows = []
    for i, (chunk, emb) in enumerate(zip(batch, embeddings)):  # pair each chunk with its embedding
        m = chunk["metadata"]  # get metadata
        rows.append({
            "doctor_id":   str(m.get("doctor_id", ""))[:50],
            "chunk_index": int(batch_start + i),
            "text":        chunk["text"][:2000],
            "name":        (m.get("name")      or "")[:200],
            "specialty":   (m.get("specialty") or "")[:300],
            "location":    (m.get("location")  or "")[:500],
            "url":         (m.get("url")       or "")[:500],
            "embedding":   emb,
        })

    col.insert(rows)

col.flush()
print(f"\nDone! {col.num_entities} vectors in Zilliz.")

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_23044\3488189411.py:58: PyMilvusDeprecationWarning: `Collection.insert` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  col.insert(rows)


































































Embedding: 100%|██████████| 66/66 [01:36<00:00,  1.46s/it]
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_23044\3488189411.py:60: PyMilvusDeprecationWarning: `Collection.flush` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  col.flush()



Done! 1045 vectors in Zilliz.


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_23044\3488189411.py:61: PyMilvusDeprecationWarning: `Collection.num_entities` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  print(f"\nDone! {col.num_entities} vectors in Zilliz.")
